# Muestreo Masivo para Caracterización de Trayectorias A320

Este módulo realiza la adquisición de datos necesaria para mapear el paisaje de energía de un Airbus A320. El objetivo es generar un dataset robusto para la posterior construcción de un modelo **QUBO**.

### 1. Estrategia de Exploración Espacial
El script utiliza un hiper-rectángulo de búsqueda centrado en el vector `x_original` ($\mathbb{R}^{10}$).
*   **Reubicación:** Al variar `x_original` entre sesiones, se exploran distintas regiones de la variedad factible.
*   **Resolución Multiescalar:** El parámetro `perturbacion` actúa como radio de búsqueda; valores pequeños permiten un muestreo de alta resolución local, mientras que valores mayores facilitan una exploración volumétrica global.

### 2. Generación Quasi-Monte Carlo (Sobol)
Se emplean **Secuencias de Sobol** con aleatorización (*scrambling*) para garantizar una cobertura óptima del espacio deca-dimensional:
*   **Muestras:** $2^{18} = 262,144$ puntos por ejecución.
*   **Ventaja:** Baja discrepancia y uniformidad superior al muestreo aleatorio, evitando clústeres de datos.

### 3. Evaluación y Persistencia de Datos
El sistema evalúa cada punto mediante la función física `phi`, filtrando singularidades (`NaN`) y gestionando el almacenamiento en `muestreo_de_puntos.csv`:
*   **Integridad:** Se utiliza una estructura de conjunto (*set*) para garantizar la **unicidad de los puntos**, evitando duplicados entre sesiones.
*   **Escritura Incremental:** El volcado inmediato al disco (`flush`) asegura la persistencia de los datos frente a interrupciones, consolidando una base de datos unificada para la regresión cuadrática.

In [ ]:
from modelo_fisico_a320 import phi
import numpy as np
import os, csv
from scipy.stats import qmc

if __name__ == "__main__":

    x_original = [221.24729842360676,239.4706178322488,222.15256959045036,
                  241.7624845156587,192.84900584897156,2.0850730239633437,
                  9.288692144473373,2.9551739109062822,1.0528001839264718,
                  7.16811807008532
                  ] # Punto validado sobre la funcion phi

    x_base = np.array(x_original)
    perturbacion = 0.0001
    l_inf = x_base - perturbacion
    l_sup = x_base + perturbacion

    # Sobol 2^18 = 262144 muestras
    sampler = qmc.Sobol(d=len(x_base), scramble=True)
    muestras = qmc.scale(sampler.random_base2(m=18), l_inf, l_sup)

    archivo_csv = "muestreo_de_puntos.csv"
    puntos_existentes = set()

    if os.path.exists(archivo_csv):
        with open(archivo_csv, 'r', newline='') as f:
            for row in csv.reader(f):
                if row:
                    try:
                        puntos_existentes.add(tuple(float(x) for x in row[:-1]))
                    except ValueError:
                        pass

    contador = len(puntos_existentes)
    print(contador, end='', flush=True)

    with open(archivo_csv, 'a', newline='') as f:
        writer = csv.writer(f)
        for x in muestras:
            c = phi(*x)
            if not np.isnan(c):
                t = tuple(x.tolist())
                if t not in puntos_existentes:
                    puntos_existentes.add(t)
                    writer.writerow(list(x) + [c])
                    f.flush()
                    contador += 1
                    print(f"\r{contador}", end='', flush=True)